# Session 5 · IoT ecosystems and platforms
## “Who pays for the SIM in year seven?”

**Blockchain and IoT · Fourth-year Computer Engineering · UNIE · Thursday, September 24, 2026**

This notebook runs **alongside the presentation**. When a slide marked **COLAB · Step N** appears, run step N here. Each step explains **what to do**, **why**, and **what to look for** in the output.

**The case for the whole session.** A maintenance company has **10,000 elevators** that currently send alerts over 2G (the fleet from session 1). It migrates them to **LTE-M** modules with SIM cards. Each elevator sends **one status message per hour** and an alert if something goes wrong. The communication module must last **15 years**.

The question is not purely technical: **when we reach year seven, who is still paying, and why?**

| Step | What you do | Minutes |
|---|---|---|
| 0 | Set up the notebook and define the case | 1 |
| 1 | Lifetimes: what ends before the elevator does | 4 |
| 2 | Platform costs (AWS versus Azure) | 5 |
| 3 | How many bytes a message actually uses | 5 |
| 4 | Will a 500 MB plan last ten years? | 5 |
| 5 | Total cost over 15 years and year seven | 5 |
| 6 | When does the business stop making economic sense? | 3 |
| 7 | Which assumption changes the total most? | 2 |
| 8 | ACT 1 worksheet, column 7 | 3 |

> **Figures.** Published rates are dated and sourced below. Classroom **working assumptions** carry a 🔧 marker. You may change them: the conclusions should hold up when assumptions change. We test this in Step 7.

---
## Step 0 · Set up the notebook · ⏱ 1 min · 🖥️ slide 6 (3:05)
**What to do:** load the libraries and set the case parameters.
**Why:** every step uses these variables. Changing one here changes the whole notebook.
**What to check:** `✅ Case loaded` at the end.

*Run with* **Shift + Enter**. In Colab, the fields on the right (`# @param`) can be edited as a form.

In [ ]:
import math, json, ssl, socket, threading, subprocess, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXPORT = os.environ.get("S5_EXPORT")          # used only by the instructor to export figures
GRANDE = bool(EXPORT)
plt.rcParams.update({
    "figure.figsize": (12, 5.6) if GRANDE else (10, 4.8),
    "font.size": 17 if GRANDE else 12,
    "axes.titlesize": 20 if GRANDE else 13,
    "axes.spines.top": False, "axes.spines.right": False,
})
def guardar(fig, nombre):
    if EXPORT:
        fig.savefig(f"{EXPORT}/{nombre}.png", dpi=170, bbox_inches="tight")

# ---------- The case ----------
N         = 10000   # @param {type:"integer"}
VIDA      = 15      # @param {type:"integer"}
MSG_DIA   = 24      # @param {type:"integer"}
AÑO_0     = 2026
SIM_MES   = 1.00    # @param {type:"number"}
# 🔧 SIM_MES: €1/month per SIM is a working assumption (monthly IoT rates are negotiated by volume).
ESCALERA  = 200     # @param {type:"number"}
# 🔧 ESCALERA: cost of a site visit (travel, labor, and parts). Working assumption.

print(f"Fleet: {N:,} elevators · module life: {VIDA} years · {MSG_DIA} messages/day each")
print(f"Fleet messages per day: {N*MSG_DIA:,}")
print(f"SIM at €{SIM_MES:.2f}/month → €{N*SIM_MES*12:,.0f} per year".replace(",", "."))
print("✅ Case loaded")

---
## Step 1 · Lifetimes · ⏱ 4 min · 🖥️ slide 27 (13:20)
**What to do:** plot how long each component required by a connected elevator lasts.
**Why:** the elevator lasts decades, but its SIM contract, network, and cloud service may not. **By year seven, some of these clocks have already stopped.**
**What to check:** which bars end **to the left** of the year-seven line.

*Duration* = years from start to end. Dated entries are **real examples**; 🔧 marks working assumptions.

In [ ]:
relojes = [
    # (item, years, category)
    ("Elevator (the asset)",                          25,  "activo"),     # 🔧 approximate magnitude
    ("Communication module (required lifetime)",       VIDA,"activo"),
    ("1NCE long-term plan (€12, 10 years)",           10,  "contrato"),
    ("AWS IoT Analytics  (Apr 2018 → Dec 2025)",      7.7, "nube"),
    ("AWS IoT Events     (May 2019 → May 2026)",      7.0, "nube"),
    ("Google Cloud IoT Core (Feb 2018 → Aug 2023)",   5.5, "nube"),
    ("Typical SIM contract (working assumption)",                      3,   "contrato"),
    ("Personal phone: updates (class assumption)",             5,   "producto"),  # 🔧 example; replace with the manufacturer’s actual support commitment
    ("Spotify Car Thing (Feb 2022 → Dec 2024)",       2.8, "producto"),
    ("2G in Spain: reported 2027–2030 shutdown",     2.5, "red"),        # measured from today; press estimates, not official dates
]
colores = {"activo":"#0B3C49", "contrato":"#F2A541", "nube":"#3E7CB1", "producto":"#C8553D", "red":"#7A7A7A"}

fig, ax = plt.subplots()
for i, (qué, años, tipo) in enumerate(relojes[::-1]):
    ax.barh(i, años, color=colores[tipo])
    ax.text(años + 0.3, i, f"{años:g} years", va="center")
ax.set_yticks(range(len(relojes))); ax.set_yticklabels([r[0] for r in relojes[::-1]])
ax.axvline(7, color="#C8553D", lw=3, ls="--"); ax.text(7.2, -0.9, "year 7", color="#C8553D", weight="bold")
ax.set_xlabel("years of life"); ax.set_xlim(0, 28); ax.set_ylim(-1.3, len(relojes) - 0.5)
ax.set_title("What ends before the elevator does?")
plt.tight_layout(); guardar(fig, "p1_relojes"); plt.show()

antes = [r[0] for r in relojes if r[1] < 7]
print(f"End before year 7: {len(antes)} of {len(relojes)}")
for a in antes: print("  ·", a)

✏️ **Your turn (1 min).** Add a row of your own: how many years of updates did your phone manufacturer promise? Copy one line from the list, change the description and years, and run it again.

**What you just saw.** Three major cloud services lasted between 5.5 and 7.7 years. A module designed for 15 years will live through **at least two generations of cloud services** and, if it used 2G, could outlive its network. The year-seven problem is that **someone must renew a commitment**.

**Student response · Step 1.** I added an example row for my phone with **five years of updates** as a working assumption (🔧), because neither its manufacturer nor its model is specified. For my actual phone, I would check the manufacturer's official update policy from its release date. The example expires before year seven, so the design must allow for software maintenance and replacement without relying on an indefinite promise.

---
## Step 2 · Platform costs · ⏱ 5 min · 🖥️ slide 43 (26:20)
**What to do:** calculate the annual AWS IoT Core and Azure IoT Hub charges for messages from 10,000 elevators.
**Why:** this is the comparison most people request first. Let us see **how much money** is at stake.
**What to check:** each annual total in euros and the last line: **platform versus SIM**.

How each service charges (rates checked August 31, 2026):
- **AWS IoT Core** charges **per message**: $1.00 per million in the first tier, counting each message in **5 KB blocks**. It also charges **$0.08 per million connected minutes**.
- **Azure IoT Hub** charges **per unit per month**: an S1 unit (€21.97/month) or B1 unit (€8.79/month) allows **400,000 messages a day**, counted in **4 KB blocks**. Exceeding that limit requires another whole unit.

In [ ]:
CAMBIO = 0.86   # @param {type:"number"}
# 🔧 CAMBIO: euros per dollar. Working assumption; use the current rate.

def factura_aws(n, msg_dia, bytes_msg=100, min_conectado_dia=1440):
    bloques   = math.ceil(bytes_msg / 5120)                    # 5 KB blocks
    mensajes  = n * msg_dia * 365 * bloques
    minutos   = n * min_conectado_dia * 365
    usd = mensajes / 1e6 * 1.00 + minutos / 1e6 * 0.08
    return usd * CAMBIO                                        # € per year

EDICIONES = {"B1": (400_000, 8.79), "S1": (400_000, 21.97), "S2": (6_000_000, 219.65), "S3": (300_000_000, 2196.55)}
def factura_azure(n, msg_dia, bytes_msg=100, edicion="S1"):
    cupo, eur_mes = EDICIONES[edicion]
    bloques  = math.ceil(bytes_msg / 4096)                     # 4 KB blocks
    unidades = math.ceil(n * msg_dia * bloques / cupo)
    return unidades * eur_mes * 12, unidades                   # € per year, units

aws = factura_aws(N, MSG_DIA)
az, uds = factura_azure(N, MSG_DIA, edicion="S1")
sim = N * SIM_MES * 12
print(f"AWS IoT Core  : €{aws:10,.0f}/year (messages + connected minutes)".replace(",", "."))
print(f"Azure IoT Hub : €{az:10,.0f}/year ({uds} S1 unit)".replace(",", "."))
print(f"Fleet SIM cost: €{sim:10,.0f}/year".replace(",", "."))
print(f"\n→ The SIM costs {sim/max(aws,az):,.0f} times as much as the more expensive platform.".replace(",", "."))

In [ ]:
# What if the fleet grows? Repeat the calculation from 1,000 to 1,000,000 devices.
parques = np.array([1e3, 3e3, 1e4, 3e4, 1e5, 3e5, 1e6])
fig, ax = plt.subplots()
ax.plot(parques, [factura_aws(p, MSG_DIA) for p in parques], "o-", lw=3, label="AWS IoT Core", color="#F2A541")
ax.plot(parques, [factura_azure(p, MSG_DIA, edicion="S1")[0] for p in parques], "s-", lw=3, label="Azure IoT Hub S1", color="#3E7CB1")
ax.plot(parques, parques * SIM_MES * 12, "-", lw=4, label=f"SIM at €{SIM_MES:.2f}/month", color="#C8553D")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("connected elevators"); ax.set_ylabel("€ per year")
ax.set_title("Platform comparison versus SIM spending")
ax.legend(); ax.grid(alpha=.3, which="both")
plt.tight_layout(); guardar(fig, "p2_plataforma"); plt.show()

✏️ **Your turn (1 min).** Change `edicion="S1"` to `"B1"` in the preceding cell. B1 is cheaper, but **cannot send commands to the elevator** or store its *device twin*. Azure does not allow an in-place downgrade from Standard to Basic: you must create another hub and register the devices again. How much do you save, and what do you lose?

**What you just saw.** With 10,000 elevators, the platform costs **hundreds of euros per year**; SIM cards cost **over one hundred thousand**. Comparing AWS and Azure still matters for other reasons (available services, their longevity, migration cost), **but the message bill is not the main expense**. This calculation excludes rules, storage, and other services, which may cost more than messages in a real bill.

In [ ]:
az_s1, u_s1 = factura_azure(N, MSG_DIA, edicion="S1")
az_b1, u_b1 = factura_azure(N, MSG_DIA, edicion="B1")
print(f"S1: €{az_s1:.2f}/year ({u_s1} unit); B1: €{az_b1:.2f}/year ({u_b1} unit)")
print(f"B1 savings: €{az_s1-az_b1:.2f}/year and €{(az_s1-az_b1)*VIDA:.2f} over {VIDA} years, assuming unchanged rates")

**Student response · Step 2.** At 240,000 messages per day, one unit suffices: S1 costs **€263.64/year**, and B1 **€105.48/year**. The saving is **€158.16/year** (€2,372.40 over 15 years). B1 lacks cloud-to-device messages and *device twins*, which we need for remote commands or configuration. The choice requires reviewing functional requirements and migration cost; the message saving is small compared with SIM costs.

---
## Step 3 · How many bytes a message actually uses · ⏱ 5 min · 🖥️ slide 52 (36:20)
**What to do:** construct an elevator message and measure the payload and **everything transmitted around it**; then measure a TLS *handshake* against a real server.
**Why:** the SIM bills for **traffic across the network**, not just payload bytes. Establishing an encrypted connection also costs data.
**What to check:** what portion of the bar is payload, and how many bytes **opening** an encrypted connection takes.

Vocabulary: **TLS** = encryption for the connection (the lock in HTTPS and secure MQTT on port 8883). **Handshake** = the initial exchange of certificates and keys between client and server. **Header** = bytes added by each network layer to deliver the message.

In [ ]:
estado = {"id": "ASC-004213", "t": 1790000000, "est": 0, "puerta": 1, "viajes": 37, "alarma": 0}
dato   = json.dumps(estado, separators=(",", ":")).encode()
TOPIC  = b"asc/004213/estado"
print(dato.decode()); print(f"Payload: {len(dato)} bytes")

# Layers of an MQTT (QoS 1) message over TLS 1.3 and TCP/IPv4
capas = {
    "payload (JSON)":               len(dato),
    "MQTT header":             2 + 2 + len(TOPIC) + 2,    # fixed header + topic length + topic + packet ID
    "TLS (encrypted record)":    22,                        # measured below: 5-byte header + 1-byte type + 16-byte tag
    "TCP/IP (outbound)":              40,                        # 20 IPv4 + 20 TCP, excluding options
    "server TCP acknowledgment":    40,
    "MQTT PUBACK (inbound)":      4 + 22 + 40,               # QoS 1 acknowledgment, also encrypted
}
BYTES_MSG = sum(capas.values())
fig, ax = plt.subplots(figsize=(13, 3.4) if GRANDE else (10, 2.8))
izq = 0
paleta = ["#0B3C49", "#3E7CB1", "#F2A541", "#9DB4C0", "#C2D3DA", "#E6B89C"]
for (k, v), c in zip(capas.items(), paleta):
    ax.barh(0, v, left=izq, color=c); ax.text(izq + v/2, 0, f"{v}", ha="center", va="center", color="white" if c in ("#0B3C49","#3E7CB1") else "black")
    izq += v
ax.set_yticks([])
ax.set_title(f"{len(dato)} payload bytes → {BYTES_MSG} network bytes ({len(dato)/BYTES_MSG:.0%} is payload)")
ax.legend(capas.keys(), ncol=3, loc="upper center", bbox_to_anchor=(.5, -0.25), frameon=False)
plt.tight_layout(); guardar(fig, "p3_capas"); plt.show()

Now measure the **connection setup cost**: count the bytes sent and received during a TLS *handshake*. First use a real Internet server; if network access is blocked, use a local server with a minimal certificate.

*(Certificate validation is disabled for this measurement. **Never** disable it on a deployed device.)*

In [ ]:
def medir_tls(host, port, sesion=None, payload=b"x" * 96):
    raw = socket.create_connection((host, port), timeout=6)
    ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    entra, sale = ssl.MemoryBIO(), ssl.MemoryBIO()
    tls = ctx.wrap_bio(entra, sale, server_hostname=host, session=sesion)
    n = {"tx": 0, "rx": 0}
    def enviar():
        d = sale.read()
        if d: raw.sendall(d); n["tx"] += len(d)
    def recibir():
        d = raw.recv(65536); n["rx"] += len(d); entra.write(d)
    while True:
        try: tls.do_handshake(); break
        except ssl.SSLWantReadError: enviar(); recibir()
    enviar(); saludo = n["tx"] + n["rx"]
    tls.write(payload); antes = n["tx"]; enviar(); registro = n["tx"] - antes - len(payload)
    raw.settimeout(1.0)
    try:
        for _ in range(3): recibir()             # collect session-resumption tickets sent by the server
    except (socket.timeout, OSError): pass
    for _ in range(3):
        try: tls.read(4096)
        except (ssl.SSLWantReadError, ssl.SSLError): break
    s = tls.session; raw.close()
    return {"saludo": saludo, "registro": registro, "version": tls.version(), "sesion": s, "reanudada": tls.session_reused}

def servidor_local(port=8883):
    subprocess.run("openssl req -x509 -newkey rsa:2048 -nodes -keyout /tmp/k.pem -out /tmp/c.pem "
                   "-days 30 -subj /CN=broker.local 2>/dev/null", shell=True, check=True)
    ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER); ctx.load_cert_chain("/tmp/c.pem", "/tmp/k.pem")
    s = socket.socket(); s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1); s.bind(("127.0.0.1", port)); s.listen(8)
    def atender(c):
        try:
            t = ctx.wrap_socket(c, server_side=True)
            while t.recv(4096): t.sendall(b"\x40\x02\x00\x01")
        except Exception: pass
    def bucle():
        while True:
            c, _ = s.accept(); threading.Thread(target=atender, args=(c,), daemon=True).start()
    threading.Thread(target=bucle, daemon=True).start(); time.sleep(0.3)

medida = None
for host, port in [("test.mosquitto.org", 8883), ("broker.hivemq.com", 8883), ("www.google.com", 443)]:
    try:
        medida = medir_tls(host, port); origen, H, P = f"{host}:{port}", host, port; break
    except Exception as e:
        print(f"· {host}:{port} did not respond ({type(e).__name__})")
if medida is None:
    servidor_local(); H, P = "127.0.0.1", 8883
    medida = medir_tls(H, P); origen = "local server with a minimal certificate"

HS_COMPLETO = medida["saludo"]
print(f"\nMeasured against {origen} ({medida['version']})")
print(f"  Full handshake : {HS_COMPLETO:,} bytes".replace(",", "."))
print(f"  TLS overhead per encrypted message: {medida['registro']} bytes")
try:
    otra = medir_tls(H, P, sesion=medida["sesion"])
    HS_REANUDADO = otra["saludo"] if otra["reanudada"] else None
except Exception:
    HS_REANUDADO = None
if HS_REANUDADO:
    print(f"  Resumed handshake (with session ticket): {HS_REANUDADO:,} bytes".replace(",", "."))
else:
    HS_REANUDADO = int(HS_COMPLETO * 0.45); print(f"  (server did not resume; assumed: {HS_REANUDADO} bytes)")
print(f"\n→ Opening the connection costs as much data as {HS_COMPLETO / BYTES_MSG:.0f} status messages.")

**Observation · Step 3.** JSON accounts for only part of total traffic: MQTT, TLS, TCP/IP, and acknowledgments also consume bytes. A TLS handshake depends on the server and its certificates, so this measurement is illustrative. In production, verify certificates, budget for reconnections, and measure actual cellular traffic, including any signaling billed by the carrier.

**What you just saw:**
1. The payload is a **minority** of transmitted data; the rest is headers and acknowledgments.
2. **Opening** an encrypted connection can cost as much as dozens of messages. A real server sends its full certificate chain and costs more than our minimal test server.
3. **Resuming** a session (the device saves a ticket from the previous handshake) reduces setup traffic considerably. A firmware decision made once affects costs for 15 years.

---
## Step 4 · Will a 500 MB plan last ten years? · ⏱ 5 min · 🖥️ slide 56 (43:20)
**What to do:** calculate annual megabytes per elevator under different **connection behaviors** and compare them with a real long-term plan: **1NCE, €12 for ten years with 500 MB** (June 2026 price list). That averages **50 MB per year**.
**Why:** the long-term plan is cheap only if the design stays within its allowance. Otherwise, you must top up (€10 per additional 500 MB) or the SIM is deactivated.
**What to check:** which bars cross the red line.

Two connection approaches:
- **Always connected.** The connection stays open and the device sends a **keepalive** (an “I'm still here” MQTT PINGREQ) every *k* seconds.
- **Sleep and reconnect.** The modem sleeps (saving battery), then opens TCP, handshakes over TLS, connects, publishes, and closes for each message.

In [ ]:
PING   = (2 + 22 + 40) * 2 + 40                  # encrypted PINGREQ + PINGRESP, TCP/IP, plus acknowledgment
TCP_AB = 3 * 60 + 4 * 52                         # TCP open (3 packets) and close (4 packets)
CONNECT= (24 + 22 + 40) + (4 + 22 + 40) + 40     # encrypted MQTT CONNECT + CONNACK (session 4)

def mb_año(modo, keepalive_s=60, msg_dia=MSG_DIA, bytes_msg=None, hs=None):
    bytes_msg = bytes_msg or BYTES_MSG
    if modo == "siempre":
        pings = 86400 / keepalive_s
        b_dia = msg_dia * bytes_msg + pings * PING
    else:  # "reconecta"
        b_dia = msg_dia * (TCP_AB + hs + CONNECT + bytes_msg)
    return b_dia * 365 / 1e6

escenarios = {
    "always · keepalive 30 s":   mb_año("siempre", 30),
    "always · keepalive 60 s":   mb_año("siempre", 60),
    "always · keepalive 5 min":  mb_año("siempre", 300),
    "always · keepalive 20 min": mb_año("siempre", 1200),
    "reconnect · full TLS handshake":   mb_año("reconecta", hs=HS_COMPLETO),
    "reconnect · resumed TLS handshake":  mb_año("reconecta", hs=HS_REANUDADO),
}
CUPO = 500 / 10   # MB/year in the long-term plan
fig, ax = plt.subplots()
nombres, valores = list(escenarios), list(escenarios.values())
ax.barh(nombres[::-1], valores[::-1], color=["#0B3C49" if v <= CUPO else "#C8553D" for v in valores[::-1]])
ax.axvline(CUPO, color="#C8553D", lw=3); ax.text(CUPO, -0.95, "  allowance: 50 MB/year (500 MB in 10 years)", color="#C8553D", weight="bold"); ax.set_ylim(-1.2, len(nombres) - 0.5)
for i, v in enumerate(valores[::-1]): ax.text(v + 1, i, f"{v:.1f} MB", va="center")
ax.set_xlabel("MB/year per elevator"); ax.set_title("Does it fit in the long-term plan?")
plt.tight_layout(); guardar(fig, "p4_cupo"); plt.show()

for k, v in escenarios.items():
    recargas = max(0, math.ceil((v * 10 - 500) / 500))
    veredicto = "fits" if v <= CUPO else f"DOES NOT FIT: {recargas} €10 top-up(s) in ten years"
    print(f"{k:28s} {v:6.1f} MB/year → {veredicto}")

✏️ **Your turn (1 min).** Find the **shortest keepalive interval** that still fits within 50 MB/year. Change the `60` on this line until you find it:

In [ ]:
mi_keepalive = 60   # @param {type:"integer"}
print(f"keepalive {mi_keepalive} s → {mb_año('siempre', mi_keepalive):.1f} MB/year  (allowance: {CUPO:.0f})")

In [ ]:
umbral_s = 86400 * PING / (CUPO * 1e6 / 365 - MSG_DIA * BYTES_MSG)
keepalive_minimo_entero = math.ceil(umbral_s)
print(f"Theoretical limit: {umbral_s:.2f} s; shortest whole-second interval that fits: {keepalive_minimo_entero} s")
print(f"At the minimum: {mb_año('siempre', keepalive_minimo_entero):.3f} MB/year; one second less: {mb_año('siempre', keepalive_minimo_entero-1):.3f} MB/year")

**What you just saw.** With the **same payload** and **same plan**, a firmware parameter (keepalive or session resumption) determines whether the €12 SIM lasts ten years or needs multiple top-ups. **A long-term plan is a byte budget, not just a price**, like yesterday's thirty-second radio budget.

---
## Step 5 · Total cost over 15 years and year seven · ⏱ 5 min · 🖥️ slide 63 (52:05)
**What to do:** add the fleet's SIM, platform, and **site-visit** costs year by year for three scenarios.
**Why:** this is **TCO** (*total cost of ownership*): the cost of keeping the system running throughout its life, rather than its purchase price.
**What to check:** cumulative spending **by year seven** in each scenario and the cost of **year seven itself**.

In [ ]:
def tco(tarifa="monthly", mb=None, año_visita=None, años=VIDA, n=N):
    mb = mb if mb is not None else mb_año("siempre", 1200)
    plataforma = factura_azure(n, MSG_DIA, edicion="S1")[0] / n          # € per elevator per year
    saldo, gasto = 0.0, []
    for a in range(años):
        g = plataforma
        if tarifa == "monthly":
            g += SIM_MES * 12
        else:                                   # long-term plan: €12 every ten years with 500 MB; €10 per 500 MB top-up
            if a % 10 == 0: g += 12; saldo += 500
            saldo -= mb
            while saldo < 0: g += 10; saldo += 500
        if año_visita is not None and a == año_visita: g += ESCALERA
        gasto.append(g * n)
    return np.array(gasto)

esc = {
    f"Monthly SIM €{SIM_MES:.2f}/month":            tco("monthly"),
    "Long-term plan · 20-min keepalive":             tco("vida", mb_año("siempre", 1200)),
    "Long-term plan · 60-s keepalive":               tco("vida", mb_año("siempre", 60)),
    "Long-term plan + mandatory visit in year 7": tco("vida", mb_año("siempre", 1200), año_visita=6),
}
fig, ax = plt.subplots()
años = np.arange(1, VIDA + 1)
for (k, g), c in zip(esc.items(), ["#C8553D", "#0B3C49", "#F2A541", "#3E7CB1"]):
    acum = np.concatenate([[0], g.cumsum()]) / 1e6
    ax.plot(np.arange(0, VIDA + 1), acum, lw=3.5, label=k, color=c)
    ax.text(VIDA + 0.2, acum[-1], f"{acum[-1]:.2f} M€", va="center", color=c, fontsize="small")
ax.axvline(7, color="#C8553D", ls="--", lw=2); ax.text(7.1, 0.05, "year 7", color="#C8553D", weight="bold")
ax.set_xlim(0, VIDA + 2.5); ax.set_xlabel("year"); ax.set_ylabel("Cumulative €m (10,000 elevators)")
ax.set_title("Total connectivity and platform cost over 15 years"); ax.legend(fontsize="small", loc="upper left", bbox_to_anchor=(0.0, 0.88)); ax.grid(alpha=.3)
plt.tight_layout(); guardar(fig, "p5_tco"); plt.show()

tabla = pd.DataFrame({k: {"cumulative through year 7 (€)": g[:7].sum(), "year 7 only (€)": g[6], "15 years (€)": g.sum()} for k, g in esc.items()}).T
tabla.round(0).astype(int)

**What you just saw:**
- The plans differ by **an order of magnitude**, depending on the design choices in Step 4.
- **A single site visit** to 10,000 elevators (`ESCALERA` 🔧 × 10,000) costs more than ten years of the long-term plan. That makes a carrier change requiring physical SIM replacement, or a network shutdown, **the** cost of year seven.
- The **IoT eSIM** (GSMA **SGP.32**) addresses this by allowing a **remote** carrier change without site visits.

---
## Step 6 · When does it stop making economic sense? · ⏱ 3 min · 🖥️ slide 70 (60:40)
**What to do:** consider the **module manufacturer's** position: it sells the device **once** and promises “connectivity included”. Calculate cumulative margin year by year.
**Why:** if the customer pays once but costs recur monthly, **a year will come when the manufacturer has a financial incentive to switch it off**. Spotify Car Thing illustrates the risk (disabled on December 9, 2024, less than three years after launch).
**What to check:** when each curve crosses zero.

In [ ]:
PRECIO   = 90    # @param {type:"number"}
COSTE_HW = 45    # @param {type:"number"}
SOPORTE  = 3     # @param {type:"number"}
CUOTA    = 2.0   # @param {type:"number"}
# 🔧 Working assumptions: selling price, manufacturing cost, annual support and cloud cost per device, monthly customer fee.

def margen(modelo, años=VIDA):
    m = [PRECIO - COSTE_HW]
    for a in range(1, años + 1):
        if modelo == "included · monthly SIM":  r = -(SIM_MES * 12 + SOPORTE)
        elif modelo == "included · long-term plan": r = -((12 if (a - 1) % 10 == 0 else 0) + SOPORTE)
        else:                                    r = CUOTA * 12 - (SIM_MES * 12 + SOPORTE)   # subscription
        m.append(m[-1] + r)
    return np.array(m)

fig, ax = plt.subplots()
for mod, c in zip(["included · monthly SIM", "included · long-term plan", "customer subscription"], ["#C8553D", "#F2A541", "#0B3C49"]):
    m = margen(mod); ax.plot(range(VIDA + 1), m, lw=3.5, label=mod, color=c)
    cruce = next((i for i, v in enumerate(m) if v < 0), None)
    print(f"{mod:28s} → negative margin {'in year ' + str(cruce) if cruce else 'never within ' + str(VIDA) + ' years'}")
ax.axhline(0, color="black", lw=1); ax.axvline(7, color="#C8553D", ls="--", lw=2)
ax.set_xlabel("year"); ax.set_ylabel("cumulative margin per device (€)")
ax.set_title("When does the manufacturer start losing money?"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); guardar(fig, "p6_margen"); plt.show()

✏️ **Your turn (1 min).** What `PRECIO` would allow “connectivity included with a monthly SIM” to last the full 15 years? What minimum `CUOTA` makes a subscription viable? Test and note both figures for the discussion.

**What you just saw.** “Included forever” has a calculable expiration date. If the contract does not state it, **the manufacturer's spreadsheet sets the date**, not the customer.

In [ ]:
precio_minimo_equilibrio = COSTE_HW + VIDA * (SIM_MES * 12 + SOPORTE)
cuota_minima_equilibrio = (SIM_MES * 12 + SOPORTE) / 12
print(f"Minimum 15-year break-even price: {precio_minimo_equilibrio:.2f} €")
print(f"Minimum break-even monthly fee: {cuota_minima_equilibrio:.2f} €/month")
print("The fee assumes a €90 upfront payment and constant costs; it excludes defaults, taxes, target margin, and inflation.")

**Student response · Step 6.** To include a monthly SIM for 15 years, the minimum break-even price is **€270**: €45 hardware + 15 × (€12 SIM + €3 support). At a €90 initial price, the break-even subscription fee is **€1.25/month** (€15 in annual recurring costs). Both cases yield zero margin; a sustainable business would require additional margin and funds for future migrations.

---
## Step 7 · Which assumption changes the bill? · ⏱ 2 min · 🖥️ slide 83 (74:55)
**What to do:** raise and lower each assumption by 50% and observe the effect on the 15-year cost per elevator.
**Why:** many numbers today are 🔧 assumptions. A conclusion is useful only if it **survives** inaccurate estimates.
**What to check:** the longest and shortest bars.

In [ ]:
def coste_ascensor(sim_mes=SIM_MES, plataforma=None, visitas=1, escalera=ESCALERA, soporte=SOPORTE):
    plataforma = factura_azure(N, MSG_DIA, edicion="S1")[0] / N if plataforma is None else plataforma
    return VIDA * (sim_mes * 12 + plataforma + soporte) + visitas * escalera

base = coste_ascensor()
pl0  = factura_azure(N, MSG_DIA, edicion="S1")[0] / N
supuestos = {
    "SIM price (€/month)":  lambda f: coste_ascensor(sim_mes=SIM_MES * f),
    "site-visit cost":       lambda f: coste_ascensor(escalera=ESCALERA * f),
    "support per device per year": lambda f: coste_ascensor(soporte=SOPORTE * f),
    "cloud platform":     lambda f: coste_ascensor(plataforma=pl0 * f),
}
filas = sorted(((k, g(0.5) - base, g(1.5) - base) for k, g in supuestos.items()), key=lambda x: x[2] - x[1])
fig, ax = plt.subplots(figsize=(12, 4.2) if GRANDE else (10, 3.6))
for i, (k, lo, hi) in enumerate(filas):
    ax.barh(i, hi, color="#C8553D"); ax.barh(i, lo, color="#0B3C49")
    ax.text(hi + 3, i, f"±{hi:.0f} €", va="center")
ax.set_yticks(range(len(filas))); ax.set_yticklabels([f[0] for f in filas]); ax.axvline(0, color="black")
ax.set_xlabel(f"15-year change in € per elevator (baseline: €{base:.0f})")
ax.set_title("What if this assumption were off by 50%?")
plt.tight_layout(); guardar(fig, "p7_sensibilidad"); plt.show()

**Student response · Step 7.** With one site visit over 15 years, the baseline is approximately **€425.40 per elevator**. A 50% change shifts the total by ±€100 for a site visit, ±€90 for the SIM, ±€22.50 for support, and ±€0.20 for the Azure platform. These are assumptions and scenarios, not a forecast for a real contract. A site visit strongly affects costs; if no visit is needed, its cost is zero.

**What you just saw.** The platform under debate produces **the shortest bar**. The SIM, site visits, and support drive the bill: **monthly charges and physical maintenance**.

---
## Step 8 · ACT 1 worksheet, column 7 · ⏱ 3 min · 🖥️ slide 91 (81:50)
**What to do:** add **column 7: who pays and for how long** to your worksheet from sessions 3 and 4, using **your own** device rather than the elevators.
**What to check:** leave no cell at “we'll figure it out later”. If you do not know a value, explain **how you would find it**.

In [ ]:
# Example worksheet: refrigerated-container (reefer) temperature sensor.
# Illustrative case; replace with your actual device from sessions 3 and 4.
FICHA_MB_ANIO = 24 * 220 * 365 / 1e6  # 🔧 24 messages/day and 220 billable bytes/message
ficha = {
    "Device (from your worksheet)": "Refrigerated-container temperature sensor (illustrative; confirm your S3/S4 device)",
    "Required lifetime (years)": 10,
    "Connectivity: technology and plan": "LTE-M with 1NCE long-term SIM: €12 / ten years, up to 500 MB; verify coverage and contract",
    "MB or airtime per year (session 4 / Step 4)": f"{FICHA_MB_ANIO:.2f} estimated MB/year, 24 messages/day × 220 B × 365 (🔧); validate with real cellular measurements",
    "Platform and pricing basis": "Azure IoT Hub S1: units/month, 400,000 messages/day per unit; shared cost and additional services billed separately",
    "Recurring cost per device per year (€)": f"~{12/10 + factura_azure(N, 24, edicion='S1')[0]/N:.2f} €/year (prorated SIM + Hub shared across 10,000 sensors; 🔧 excludes maintenance and storage)",
    "Who pays (contract, not assumption)": "Proposed contract: logistics operator pays for connectivity and cloud; supplier maintains firmware and support. Verify clauses and term",
    "Which commitment expires first (Step 1)": "Connectivity or platform contract/SLA: specify expiry in the contract; review LTE-M coverage each year",
    "What happens when that commitment expires": "Telemetry and alerts stop without renewal; buffer locally, warn before expiry, and agree on export/migration",
    "Exit plan: change carrier/cloud without site visits": "Remote IoT eSIM SGP.32 provisioning if supported by hardware and carrier; standard MQTT, data export, and rotatable credentials",
}
for k, v in ficha.items():
    print(f"{k:62s} | {v}")
vacías = [k for k, v in ficha.items() if v in ("", None)]
print(f"\n{'✅ Column 7 complete' if not vacías else f'{len(vacías)} cells still missing'}")


**Engineering and blockchain judgment.** Blockchain does not replace connectivity, SIM cards, or platforms. It would make sense only if multiple operators needed to share a verifiable record of events or payments without a single trusted authority; the design would then need to justify cost, privacy, transaction volume, and key management. For this case, an audited database and clear contracts may suffice.

---
### Looking ahead to session 6 (Wednesday, September 30)
We have seen that **someone must keep paying** for fifteen years. That works only if the system **delivers** more value than it costs. Next question: **“How many OEE points does your project improve, and how quickly?”**

### Sources (checked September 2026)
- 1NCE, price list (June 2026): Lifetime Flat €12 / 500 MB + 250 SMS / ten years; €10 / 500 MB top-up; €12 extension. https://www.1nce.com/en-us/1nce-connect/pricing · Data depletion condition (18 months to top up): https://www.1nce.com/en-us/1nce-connect/features/500-mb-data-volume
- Azure IoT Hub: quotas and metering blocks (B1/S1: 400,000 messages/day, 4 KB; Free: 8,000/day, 0.5 KB; Basic has no cloud-to-device messages or device twins; Standard cannot be downgraded to Basic). https://azure.microsoft.com/en-us/pricing/details/iot-hub/ · Western Europe euro prices checked August 31, 2026.
- AWS IoT Core: $1.00/million messages (first tier), 5 KB blocks, $0.08/million connected minutes (August 31, 2026). https://aws.amazon.com/iot-core/pricing/
- AWS IoT Events: available since May 2019 (https://aws.amazon.com/about-aws/whats-new/2019/05/aws-iot-events-now-generally-available/), retired May 20, 2026. AWS IoT Analytics: available since April 2018, retired December 15, 2025.
- Google Cloud IoT Core: available February 21, 2018 (https://techcrunch.com/2018/02/21/googles-cloud-iot-core-is-now-generally-available/), retired August 16, 2023.
- Spotify Car Thing: disabled December 9, 2024; refunds after a class-action lawsuit. https://techcrunch.com/2024/05/30/spotify-begins-offering-car-thing-refunds-as-it-faces-lawsuit-over-bricking-the-streaming-device
- GSMA SGP.32 (IoT eSIM). https://www.gsma.com/solutions-and-impact/technologies/esim/gsma_resources/sgp-32-v1-3/
- 2G/3G shutdown in Spain: there is no official 2G shutdown date; press reports about carrier-specific dates disagree. https://ecosistemastartup.com/masorange-apaga-el-3g-impacto-en-iot-y-5g-en-espana/